In [ ]:
pip install pandas pyfaidx Bio pyarrow fastparquet

In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:\variant_data\uniprot_vep_input.parquet"
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", f"SpliceAI,snv={DOCKER_RESOURCES}/spliceai_scores.raw.snv.ensembl_mane_v1.4.grch38.vcf.gz",
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,SYMBOL,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,SpliceAI_pred,"
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' (sẽ tạo ra list gồm 9 phần tử)
        split_sa = merged_chunk['SpliceAI_pred'].str.split('|', expand=True)
        
        # Lấy đúng 4 cột điểm Delta Score (Vị trí index 1, 2, 3, 4)
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:\variant_data\uniprot_vep_input.parquet...
[*] Tổng số variant: 48,779. Số lượng chunks: 1


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:/variant_data/uniprot_all_after_vep.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 48779)...

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:/variant_data/uniprot_all_after_vep.parquet


In [4]:
import pandas as pd
final_df = pd.read_parquet(r"D:/variant_data/uniprot_all_after_vep.parquet")

In [5]:
final_df['Consequence'].value_counts()

Consequence
missense_variant                                                                                          41480
downstream_gene_variant                                                                                    2120
missense_variant,splice_region_variant                                                                     1687
upstream_gene_variant                                                                                      1099
synonymous_variant                                                                                         1075
stop_gained                                                                                                 606
intron_variant                                                                                              336
non_coding_transcript_exon_variant                                                                          131
3_prime_UTR_variant                                                                         

In [6]:
final_df.isnull().sum()

Variant_ID                    0
dbSNP                         0
gene                          0
protein_AC                    0
aa_change                     0
disease                       0
CHROM                         0
POS                           0
REF                           0
ALT                           0
Label                         0
Location                      0
Allele                        0
Gene                          0
SYMBOL                        0
Feature                       0
Feature_type                  0
Consequence                   0
cDNA_position                 0
CDS_position                  0
Protein_position              0
Amino_acids                   0
Codons                        0
MANE_SELECT                   0
CANONICAL                     0
AF                            0
gnomADe_AF                    0
SpliceAI_pred                 0
phyloP100way_vertebrate       0
phyloP470way_mammalian        0
phyloP17way_primate           0
phastCon

In [7]:
import numpy as np

# Danh sách giá trị cần kiểm tra
invalid_values = ["-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)
final_df.isnull().sum()

Variant_ID                        0
dbSNP                             0
gene                             12
protein_AC                        0
aa_change                         0
disease                       32280
CHROM                             0
POS                               0
REF                               0
ALT                               0
Label                             0
Location                          0
Allele                            2
Gene                              1
SYMBOL                           94
Feature                           1
Feature_type                      1
Consequence                       0
cDNA_position                  3607
CDS_position                   3854
Protein_position               3854
Amino_acids                    3855
Codons                         3854
MANE_SELECT                     507
CANONICAL                        23
AF                            31609
gnomADe_AF                    13747
SpliceAI_pred               

In [8]:
final_df = final_df[final_df['Consequence'] == 'missense_variant']
final_df.isnull().sum()

Variant_ID                        0
dbSNP                             0
gene                              0
protein_AC                        0
aa_change                         0
disease                       27886
CHROM                             0
POS                               0
REF                               0
ALT                               0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            2
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                     257
CANONICAL                        16
AF                            26475
gnomADe_AF                    11679
SpliceAI_pred               

In [9]:
final_df = final_df.dropna(subset=['Consequence', 'SYMBOL', 'MANE_SELECT', 'CANONICAL'])
final_df.isnull().sum()

Variant_ID                        0
dbSNP                             0
gene                              0
protein_AC                        0
aa_change                         0
disease                       27646
CHROM                             0
POS                               0
REF                               0
ALT                               0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                         0
AF                            26329
gnomADe_AF                    11607
SpliceAI_pred               

In [10]:
final_df.describe()

,POS,Label,SpliceAI_pred
count,4.122200e+04,41222.000000,0.0
mean,7.484234e+07,0.328708,NaN
std,5.806486e+07,0.469749,NaN
min,4.714300e+04,0.000000,NaN
25%,3.125845e+07,0.000000,NaN
50%,5.642288e+07,0.000000,NaN
75%,1.128382e+08,1.000000,NaN
max,2.488559e+08,1.000000,NaN


In [11]:
final_df['AF'].astype(float).describe()

count    14893.000000
mean         0.143643
std          0.222904
min          0.000000
25%          0.003400
50%          0.034000
75%          0.187100
max          1.000000
Name: AF, dtype: float64

In [12]:
final_df['AF'] = final_df['AF'].astype(float)
final_df['AF'][final_df['AF'] == 0].count()

np.int64(42)

In [13]:
final_df[final_df['AF'].isnull() & final_df['gnomADe_AF'].isnull()][['AF', 'gnomADe_AF']]

,AF,gnomADe_AF
4,NaN,NaN
26,NaN,NaN
67,NaN,NaN
69,NaN,NaN
73,NaN,NaN
...,...,...
48760,NaN,NaN
48761,NaN,NaN
48764,NaN,NaN
48771,NaN,NaN


In [14]:
final_df.dropna(subset=['GERP_92_mammals'], how='all', inplace=True)
final_df.dropna(subset=['phyloP470way_mammalian'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_RS'], how='all', inplace=True)
final_df.dropna(subset=['GERP++_NR'], how='all', inplace=True)

In [16]:
final_df.isnull().sum()

Variant_ID                        0
rsid                              0
CHROM                             0
POS                               0
REF                               0
ALT                               0
gene                              0
clnsig                            0
clnhgvs                           0
clndn                             0
clnrevstat                        0
clnacc                            0
Source                            0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                   

In [15]:
final_df.replace({'AF': {np.nan: 0}}, inplace=True)
final_df.replace({'gnomADe_AF': {np.nan: 0}}, inplace=True)

In [16]:
final_df.isnull().sum()

Variant_ID                        0
dbSNP                             0
gene                              0
protein_AC                        0
aa_change                         0
disease                       16492
CHROM                             0
POS                               0
REF                               0
ALT                               0
Label                             0
Location                          0
Allele                            0
Gene                              0
SYMBOL                            0
Feature                           0
Feature_type                      0
Consequence                       0
cDNA_position                     0
CDS_position                      0
Protein_position                  0
Amino_acids                       0
Codons                            0
MANE_SELECT                       0
CANONICAL                         0
AF                                0
gnomADe_AF                        0
SpliceAI_pred               

In [17]:
final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']] = final_df[['gnomADe_AF', 'phyloP100way_vertebrate', 'phyloP470way_mammalian', 'phyloP17way_primate', 'phastCons100way_vertebrate', 'phastCons470way_mammalian', 'phastCons17way_primate', 'GERP++_RS', 'GERP++_NR', 'GERP_92_mammals']].astype(float)

In [18]:
final_df.describe()

,POS,Label,AF,gnomADe_AF,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
count,2.138100e+04,21381.000000,21381.00000,21381.000000,0.0,21381.000000,21381.000000,21381.000000,21381.000000,21381.000000,21381.000000,21381.000000,21381.000000,21381.000000
mean,7.706863e+07,0.227679,0.05787,0.057073,NaN,2.825788,3.511667,0.219199,0.629959,0.650677,0.599047,2.588395,4.989957,-0.088502
std,5.749098e+07,0.419344,0.15942,0.163244,NaN,3.168780,4.690130,0.703368,0.453750,0.457372,0.411641,3.318314,0.898632,2.245693
min,1.176660e+05,0.000000,0.00000,0.000000,NaN,-20.000000,-20.000000,-6.965000,0.000000,0.000000,0.000000,-12.300000,0.046500,-16.500000
25%,3.293358e+07,0.000000,0.00000,0.000000,NaN,0.346000,0.370000,-0.141000,0.010000,0.002000,0.101000,1.270000,4.630000,-1.780000
50%,6.281349e+07,0.000000,0.00000,0.000010,NaN,1.923000,3.074000,0.599000,0.989000,0.999000,0.807000,3.640000,5.200000,0.520000
75%,1.131176e+08,0.000000,0.01120,0.003729,NaN,5.105000,7.512000,0.665000,1.000000,1.000000,0.987000,4.960000,5.620000,1.720000
max,2.488559e+08,1.000000,1.00000,1.000000,NaN,10.003000,11.936000,0.756000,1.000000,1.000000,1.000000,6.170000,6.170000,8.790000


In [19]:
final_df.to_parquet(r'D:/variant_data/uniprot_all_after_vep_final.parquet', index=False)

In [20]:
df = pd.read_parquet(r'D:/variant_data/uniprot_all_after_vep_final.parquet')
df

,Variant_ID,dbSNP,gene,protein_AC,aa_change,disease,CHROM,POS,REF,ALT,...,SpliceAI_pred,phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,GERP++_RS,GERP++_NR,GERP_92_mammals
0,1_1041648_G_A,rs138031468,AGRN,O00468,p.Ala375Ser,None,1,1041648,G,A,...,NaN,1.487,9.713,0.648,0.021,1.000,0.018,4.660,4.66,1.320
1,1_1046922_C_A,rs149159118,AGRN,O00468,p.Thr1118Lys,None,1,1046922,C,A,...,NaN,3.499,7.506,0.599,0.998,1.000,0.677,4.180,4.18,1.140
2,1_1047863_C_A,rs142620337,AGRN,O00468,p.Pro1240Leu,None,1,1047863,C,A,...,NaN,4.040,7.453,0.599,0.997,1.000,0.816,4.100,4.10,1.330
3,1_1180851_T_C,rs13374146,TTLL10,Q6ZVT0,p.Val249Ala,None,1,1180851,T,C,...,NaN,0.026,0.917,-0.177,0.000,0.000,0.000,1.280,3.24,-2.280
4,1_1185051_G_A,rs1320571,TTLL10,Q6ZVT0,p.Ser448Asn,None,1,1185051,G,A,...,NaN,5.707,7.682,-0.040,1.000,1.000,0.009,4.540,4.54,0.527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21376,X_154999538_A_C,rs944567323,F8,P00451,p.Leu69Pro,Hemophilia A (HEMA) [MIM:306700],X,154999538,A,C,...,NaN,6.621,10.926,0.691,1.000,1.000,0.997,5.000,5.00,1.450
21377,X_155077194_A_G,rs28997578,BRCC3,P46736,p.Ile74Val,None,X,155077194,A,G,...,NaN,5.810,6.802,0.751,1.000,1.000,0.998,5.290,5.29,0.732
21378,X_155511701_C_G,rs869320708,TMLHE,Q9NVH6,p.Asp244His,"Autism, X-linked 6 (AUTSX6) [MIM:300872]",X,155511701,C,G,...,NaN,6.783,6.153,0.508,1.000,1.000,0.997,3.640,3.64,0.883
21379,Y_5100614_T_A,rs2563389,PCDH11Y,Q9BZA8,p.Asn1012Lys,None,Y,5100614,T,A,...,NaN,0.960,-0.055,-0.642,1.000,0.146,0.686,1.430,2.35,0.142


In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:\variant_data\uniprot_all_after_vep_final_spliceai.parquet"
OUTPUT_PARQUET = r"D:\variant_data\uniprot_all_after_vep_final_spliceai_full.parquet" # File xuất ra
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources"

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================

def sort_by_chromosome(df):
    """Sắp xếp DataFrame theo đúng thứ tự sinh học: 1->22, X, Y, MT và sắp xếp cả POS."""
    print("Đang sắp xếp dữ liệu theo Chromosome và Position...")
    
    # Chuẩn hóa format CHROM (Xóa chữ 'chr' để dễ map)
    df['CHROM'] = df['CHROM'].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    
    # Tạo từ điển quy đổi Chromosome thành số thứ tự
    chrom_order = {str(i): i for i in range(1, 23)}
    chrom_order.update({'X': 23, 'Y': 24, 'MT': 25, 'M': 25})
    
    # Tạo cột tạm để sắp xếp
    df['CHROM_num'] = df['CHROM'].map(chrom_order)
    
    # Những nhiễm sắc thể lạ (alt contigs, unplaced) sẽ bị đẩy xuống cuối cùng
    df['CHROM_num'] = df['CHROM_num'].fillna(99)
    
    # Sort theo CHROM_num, sau đó tới POS (Vị trí) để chuẩn form VCF nhất
    df = df.sort_values(by=['CHROM_num', 'POS'], ascending=[True, True]).reset_index(drop=True)
    
    # Dọn dẹp cột tạm
    df = df.drop(columns=['CHROM_num'])
    
    print("Hoàn tất sắp xếp!")
    return df

def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "--format", "vcf",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", "GRCh38",
        "--cache",
        "--offline",
        "--fasta",
        f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab",
        "--force_overwrite",
        "--fork", "4",
        # Chọn transcript
        "--mane_select",
        "--canonical",
        "--pick",
        "--symbol",
        # HGVS
        "--hgvs",
        # Protein annotation
        "--protein",
        # Chỉ lấy những field cần thiết
        "--fields",
        (
            "Uploaded_variation,"
            "ENSP,"
            "HGVSc,"
            "HGVSp"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        result = subprocess.run(
            vep_cmd,
            check=True,
            capture_output=True,
            text=True
        )
        print(f"[VEP CHUNK {chunk_idx} THÀNH CÔNG]")

    except subprocess.CalledProcessError as e:
        print("\n" + "=" * 70)
        print(f"[VEP ERROR - CHUNK {chunk_idx}]")
        print("=" * 70)
        print("\nReturn code:")
        print(e.returncode)
        print("\nSTDERR:")
        print(e.stderr if e.stderr else "<empty>")
        raise SystemExit(f"Dừng pipeline do VEP failed at chunk {chunk_idx}")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' 
        split_sa = merged_chunk['SpliceAI_pred'].astype('string').str.split('|', expand=True)
        # Lấy đúng 4 cột điểm Delta Score
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
df_full = sort_by_chromosome(df_full)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:\variant_data\uniprot_all_after_vep_final_spliceai.parquet...
Đang sắp xếp dữ liệu theo Chromosome và Position...
Hoàn tất sắp xếp!
[*] Tổng số variant: 21,282. Số lượng chunks: 1


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:\variant_data\uniprot_all_after_vep_final_spliceai_full.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/1 (Rows: 0 - 21282)...
[VEP CHUNK 1 THÀNH CÔNG]

[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: D:\variant_data\uniprot_all_after_vep_final_spliceai_full.parquet
